# Day 2 — Cross-Validation

#### The Logistic Regressions model from `Notebook3.3` will be used to complete todays tasks.

In [71]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

The dataset that will be used is the Bank Customer Prediction Dataset that classifies customers into churned/stayed. This dataset consist of 10000 rows and 11 columns.

In [72]:
Bank_data = pd.read_csv("Bank_Customer_Churn_Prediction.csv")
Bank_data.shape

(10000, 11)

In [73]:
Bank_data = pd.get_dummies(Bank_data,columns=['country'], drop_first=True)
Bank_data['gender'] = Bank_data['gender'].map({'Male': 0, 'Female': 1})

The 'gender' column is a categorical column which was interpretted into numbers using one-hot encoding to fit into the model.

In [74]:
x = Bank_data.drop(["churn"], axis = 1)
y = Bank_data["churn"]

#### Logistic Regression Model Training and Scaling

In [75]:

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_Scaled = scaler.transform(X_test)
Bank_data.shape


(10000, 12)

In [76]:
lr = LogisticRegression(max_iter=5000, class_weight='balanced', solver='liblinear')
lr.fit(X_train_scaled, y_train)
predictions = lr.predict(X_test_Scaled)
probabilities = lr.predict_proba(X_test_Scaled) 

#### K-Fold 

Our data will be divided into 5 folds and the f1 scores for eacah will be calculated, in addition to the mean score and standard deviation.

In [77]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(lr, X_train_scaled, y_train, cv=5, scoring="f1")
print(scores)
print(scores.mean())
print(scores.std())

[0.46875    0.51049724 0.51557465 0.48684211 0.48504983]
0.49334276552645173
0.017345708715269224


    As the results show, each fold has shown an improvement in the score which indicates that the model is progressively improving, against the single split, one fold after the other
    The mean in comparison to the standard deviation is high; the model is consistently accurate.

#### Cross Validation VS Single-split 

    Now, comparing the two approaches, i have observed that the single split score was 0.29 and after cross validation it continued to grow, with minor fluctuations, until it peaked at 0.51. These results prove that Cross Validation provides more accuracy and precision training models even though the overall performance remains weak.

#### Why Stratified k-Folds Matter?

For better understanding, let's see what are the outcomes of using Stratified K-Folds

In [78]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf_scores = cross_val_score(lr, X_train_scaled, y_train, cv= skf, scoring="f1")
print(skf_scores)
print(skf_scores.mean())
print(skf_scores.std())

[0.48144221 0.50391937 0.47374062 0.49496081 0.49894292]
0.4906011848191792
0.011261708806961522


    The k-fold created folds typically result in unequal distribution of data accross folds which is solved through stratification. Each fold of the data will have the original class balance.
    From the above results, we notice that the result show tight and consistente F1 scores reducing the spread of scores: meaning the model is more uniform, that is evident through the low standard deviation compared to the k-fold STD.